# DDPG Navigation — segway_1d_wheel

Deep Deterministic Policy Gradient for the 1D segway navigation task.

**DDPG vs SAC:**

| | SAC | DDPG |
|---|---|---|
| Policy | Stochastic (Gaussian) | **Deterministic** |
| Exploration | Entropy bonus (automatic) | **Gaussian noise added to action** |
| Critics | Twin Q-networks | **Single Q-network** |
| Target nets | Critic only | **Actor + Critic** |
| Sample efficiency | High | Moderate |
| Stability | More stable | Less stable |


In [ ]:
import mujoco, numpy as np, torch, torch.nn as nn, torch.optim as optim
import os, imageio, time, json
from PIL import Image as PILImage, ImageDraw
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
#heyyyy
XML_PATH = r"C:\Users\edward\OneDrive - City University of Hong Kong\uw\me569\owsbr\Ed\model\segway_1d_wheel.xml"  # <-- update this

SAVE_DIR     = "SavedSeeds"
TRAIN_SEEDS  = [42]
VERIFY_SEEDS = [788, 999, 555, 321, 444]
os.makedirs(SAVE_DIR, exist_ok=True)

TORQUE_MAX   =  5.0
GROUND_Z     =  0.075
X_NORM       =  2.4
XD_NORM      =  5.0
TH_NORM      =  1.57
THD_NORM     =  5.0
X_DONE_LIMIT =  2.8
TH_DONE      =  0.5

# DDPG exploration noise (matches Arthur's Ozzy version)
NOISE_START  =  0.3    # noise std at start (fraction of TORQUE_MAX)
NOISE_END    =  0.1    # noise std at end
NOISE_DECAY  =  1_000_000  # steps over which noise decays

UPDATE_EVERY    = 4
UPDATE_PER_STEP = 2


## Inline RunLogger


In [3]:
class RunLogger:
    def __init__(self, algo, seed, log_every_steps=2000):
        self.algo=algo; self.seed=seed; self.log_every_steps=log_every_steps
        self.rows=[]; self._ep_rewards=[]; self._ep_arrives=[]; self._ep_losses=[]
        self._next_log=log_every_steps; self._t0=time.time()
    def episode_end(self, total_steps, ep_count, ep_ret, arrived, loss):
        self._ep_rewards.append(ep_ret); self._ep_arrives.append(float(arrived))
        self._ep_losses.append(loss)
        avg=float(np.mean(self._ep_rewards[-100:])); rp=float(np.mean(self._ep_arrives[-100:]))*100.
        if total_steps>=self._next_log:
            self.rows.append({"steps":total_steps,"avg_reward":avg,
                              "loss":float(np.mean(self._ep_losses[-100:])),"wall_time":time.time()-self._t0})
            self._next_log+=self.log_every_steps
        return avg, rp
    def save(self, save_dir, tag, eval_results=None):
        path=f"{save_dir}/{self.algo}_{tag}.json"
        with open(path,"w") as f:
            json.dump({"algo":self.algo,"seed":self.seed,"rows":self.rows,
                       "eval":eval_results,"total_wall_time":self.total_wall_time},f,indent=2)
        print(f"Log saved: {path}")
    @property
    def total_wall_time(self): return time.time()-self._t0


## Observation helpers

Same as SAC — fast inline quaternion, no scipy.


In [4]:
def get_obs(data):
    qw,qx,qy,qz = data.qpos[3],data.qpos[4],data.qpos[5],data.qpos[6]
    theta = float(np.arcsin(np.clip(2.0*(qw*qy - qz*qx), -1.0, 1.0)))
    return np.array([data.qpos[0], data.qvel[0], theta, data.qvel[4]], dtype=np.float32)

def normalize_obs(obs):
    x,xd,th,thd = obs
    return np.array([
        np.clip(x  /X_NORM,  -1,1),
        np.clip(xd /XD_NORM, -1,1),
        np.clip(th /TH_NORM, -1,1),
        np.clip(thd/THD_NORM,-1,1),
    ], dtype=np.float32)


## Reward function

Same as SAC and PPO.


In [5]:
def nav_reward(obs, prev_x, goal_x, at_goal, done, step):
    x,xd,theta,thd=obs
    if done:    return -20.0
    if at_goal: return  50.0
    r=(abs(prev_x-goal_x)-abs(x-goal_x))*15.0
    r-=abs(x-goal_x)*0.05
    r+=0.02
    if abs(theta)>0.20: r-=(abs(theta)-0.2)*5.0
    return float(r)


## DDPG Networks

**Actor** — deterministic: outputs a single action directly (no sampling).
**Critic** — single Q-network: takes `(state, action)` → value.
Both have **target network** copies updated via soft (Polyak) averaging.


In [6]:
class DDPGActor(nn.Module):
    """Deterministic actor: state → action (no sampling, no log_std)."""
    def __init__(self, torque_max=TORQUE_MAX):
        super().__init__(); self.torque_max=torque_max
        self.net = nn.Sequential(
            nn.Linear(5,256), nn.ReLU(),
            nn.Linear(256,256), nn.ReLU(),
            nn.Linear(256,1),
        )
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.orthogonal_(m.weight,0.5); nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.net[-1].weight, 0.01)

    def forward(self, x):
        return torch.tanh(self.net(x)) * self.torque_max

    def get_action(self, obs, goal_x, deterministic=False, noise_sigma=0.0):
        dist_norm=float(np.clip((goal_x-obs[0])/5.,-1,1))
        inp=torch.FloatTensor([*normalize_obs(obs),dist_norm]).unsqueeze(0)
        with torch.no_grad():
            torque=self(inp).item()
        if not deterministic and noise_sigma>0.0:
            torque += noise_sigma * TORQUE_MAX * np.random.randn()
            torque = float(np.clip(torque, -self.torque_max, self.torque_max))
        return torque, None, None


class DDPGCritic(nn.Module):
    """Single Q-network. Input: 5-dim state + 1-dim action = 6 total."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(6,256), nn.ReLU(),
            nn.Linear(256,256), nn.ReLU(),
            nn.Linear(256,1),
        )
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.orthogonal_(m.weight,0.5); nn.init.zeros_(m.bias)

    def forward(self, obs_t, act_t):
        return self.net(torch.cat([obs_t,act_t],dim=-1))


## Replay Buffer

Same as SAC.


In [7]:
class ReplayBuffer:
    def __init__(self,capacity=1_000_000,obs_dim=5):
        self.cap=capacity; self.ptr=self.size=0
        self.obs=np.zeros((capacity,obs_dim),dtype=np.float32)
        self.act=np.zeros((capacity,1),dtype=np.float32)
        self.rew=np.zeros((capacity,1),dtype=np.float32)
        self.obs2=np.zeros((capacity,obs_dim),dtype=np.float32)
        self.done=np.zeros((capacity,1),dtype=np.float32)
    def push(self,o,a,r,o2,d):
        self.obs[self.ptr]=o; self.act[self.ptr]=[a]
        self.rew[self.ptr]=r; self.obs2[self.ptr]=o2; self.done[self.ptr]=d
        self.ptr=(self.ptr+1)%self.cap; self.size=min(self.size+1,self.cap)
    def sample(self,n=256):
        i=np.random.randint(0,self.size,n)
        return (torch.FloatTensor(self.obs[i]),torch.FloatTensor(self.act[i]),
                torch.FloatTensor(self.rew[i]),torch.FloatTensor(self.obs2[i]),
                torch.FloatTensor(self.done[i]))
    def __len__(self): return self.size


## DDPG `train_nav`

### Key differences from SAC:

| | SAC | DDPG |
|---|---|---|
| Actor update | maximise Q − α·entropy | maximise Q directly |
| Critic update | twin Q, entropy target | single Q |
| Exploration | automatic via entropy | **Gaussian noise on action** |
| Noise | none needed | sigma decays from 0.5 → 0.05 |
| Target nets | critic only | **actor + critic** |
| Alpha tuning | yes | **no** |


In [8]:
def train_nav(x_start=0.0, x_goal=2.0,
              seed=42, tag="nav_ddpg_1d",
              max_steps=2_000_000):
    torch.manual_seed(seed); np.random.seed(seed)

    GAMMA, TAU      = 0.99, 0.005
    LR_ACTOR        = 1e-4
    LR_CRITIC       = 3e-4
    BATCH, WARMUP   = 256, 5000
    MAX_EP_STEPS    = 3000

    actor      = DDPGActor(torque_max=TORQUE_MAX)
    actor_tgt  = DDPGActor(torque_max=TORQUE_MAX)
    actor_tgt.load_state_dict(actor.state_dict())
    for p in actor_tgt.parameters(): p.requires_grad=False

    critic     = DDPGCritic()
    critic_tgt = DDPGCritic()
    critic_tgt.load_state_dict(critic.state_dict())
    for p in critic_tgt.parameters(): p.requires_grad=False

    actor_opt  = optim.Adam(actor.parameters(),  lr=LR_ACTOR)
    critic_opt = optim.Adam(critic.parameters(), lr=LR_CRITIC)
    replay     = ReplayBuffer(1_000_000, obs_dim=5)

    model = mujoco.MjModel.from_xml_path(XML_PATH)
    data  = mujoco.MjData(model)

    def reset_env():
        mujoco.mj_resetData(model,data)
        data.qpos[0]=x_start; data.qpos[2]=GROUND_Z
        data.qpos[3:7]=[1,0,0,0]; data.qvel[:]=0.0
        data.qvel[4]=np.random.uniform(-0.01,0.01)
        mujoco.mj_forward(model,data); return get_obs(data)

    def env_step(torque):
        u=float(np.clip(torque,-TORQUE_MAX,TORQUE_MAX))
        data.ctrl[0]=-u
        mujoco.mj_step(model,data)
        obs=get_obs(data); x,_,th,_=obs
        done=abs(x)>X_DONE_LIMIT or abs(th)>TH_DONE
        at_goal=abs(x-x_goal)<0.10 and abs(th)<0.25
        return obs,done,at_goal

    def make_inp(obs):
        return np.array([*normalize_obs(obs),float(np.clip((x_goal-obs[0])/5.,-1,1))],dtype=np.float32)

    def ddpg_update():
        if len(replay)<BATCH: return 0.0
        ob,ac,re,ob2,dn = replay.sample(BATCH)
        with torch.no_grad():
            next_ac=actor_tgt(ob2)
            qt=critic_tgt(ob2,next_ac)
            y=re+GAMMA*(1-dn)*qt
        q=critic(ob,ac)
        cl=nn.MSELoss()(q,y)
        critic_opt.zero_grad(); cl.backward()
        nn.utils.clip_grad_norm_(critic.parameters(),1.0); critic_opt.step()
        al=-critic(ob,actor(ob)).mean()
        actor_opt.zero_grad(); al.backward()
        nn.utils.clip_grad_norm_(actor.parameters(),1.0); actor_opt.step()
        with torch.no_grad():
            for p,pt in zip(actor.parameters(),actor_tgt.parameters()):
                pt.data.mul_(1-TAU); pt.data.add_(TAU*p.data)
            for p,pt in zip(critic.parameters(),critic_tgt.parameters()):
                pt.data.mul_(1-TAU); pt.data.add_(TAU*p.data)
        return cl.item()

    logger=RunLogger("DDPG",seed,log_every_steps=2000)
    rewards=[]; losses=[]; best_avg=-9999; best_weights=None
    total_steps=ep_count=0; recent_rewards=[]

    hdr="   Steps |    Ep |    Avg10 |   Avg100 |   Rec% |   Noise |    CritL"
    print(f"\nDDPG | A={x_start}m -> B={x_goal}m | budget={max_steps:,} steps")
    print(hdr); print("-"*len(hdr))

    while total_steps<max_steps:
        obs=reset_env(); ep_ret=0.0; prev_x=obs[0]
        ep_step=0; ep_cls=[]; arrived_flag=False
        # noise decays linearly from NOISE_START to NOISE_END over NOISE_DECAY steps
        noise_sigma=max(NOISE_END, NOISE_START-(NOISE_START-NOISE_END)*min(1.0,total_steps/NOISE_DECAY))

        for _ in range(MAX_EP_STEPS):
            if total_steps<WARMUP:
                torque=np.random.uniform(-TORQUE_MAX,TORQUE_MAX)
            else:
                torque,_,_=actor.get_action(obs,x_goal,deterministic=False,noise_sigma=noise_sigma)
            obs2,done,at_goal=env_step(torque)
            ep_step+=1; total_steps+=1
            r=nav_reward(obs2,prev_x,x_goal,at_goal,done,ep_step)
            ep_ret+=r; prev_x=obs2[0]
            replay.push(make_inp(obs),torque,r,make_inp(obs2),float(done or at_goal))
            if total_steps>=WARMUP and total_steps%UPDATE_EVERY==0:
                for _ in range(UPDATE_PER_STEP): ep_cls.append(ddpg_update())
            obs=obs2
            if done or at_goal or ep_step>=MAX_EP_STEPS: arrived_flag=at_goal; break
            if total_steps>=max_steps: break

        ep_count+=1
        ep_loss=float(np.mean(ep_cls)) if ep_cls else 0.0
        rewards.append(ep_ret); losses.append(ep_loss); recent_rewards.append(ep_ret)
        avg100,rp=logger.episode_end(total_steps,ep_count,ep_ret,arrived_flag,ep_loss)

        if ep_count%10==0:
            avg10=float(np.mean(recent_rewards[-10:]))
            icon="OK" if avg100>0 else "UP" if avg100>-20 else ".."
            print(f"{total_steps:>8,} | {ep_count:>5} | {avg10:>8.2f} | {avg100:>8.2f} | "
                  f"{rp:>5.1f}% | {noise_sigma:>7.3f} | {ep_loss:>8.3f}  {icon}",flush=True)
            if avg100>best_avg:
                best_avg=avg100
                best_weights={k:v.clone() for k,v in actor.state_dict().items()}
                torch.save(actor.state_dict(),f"{SAVE_DIR}/nav_ddpg_{tag}.pth")

    if best_weights: actor.load_state_dict(best_weights)
    return actor,rewards,losses,logger


In [9]:
def train_one_verify_many(algo_label, train_seed=42,
                          verify_seeds=(788,999,555,321,444),
                          x_start=0.0, x_goal=2.0, step_budget=500_000):
    verify_seeds=list(verify_seeds)
    print(f"\n{60*chr(61)}\n  {algo_label}  TRAIN seed={train_seed}\n{60*chr(61)}")
    net,rewards,losses,logger=train_nav(x_start=x_start,x_goal=x_goal,
        seed=train_seed,tag=f"cmp_seed{train_seed}",max_steps=step_budget)

    def eval_policy(net,seed,n=20):
        torch.manual_seed(seed); np.random.seed(seed)
        m=mujoco.MjModel.from_xml_path(XML_PATH); d=mujoco.MjData(m)
        strict=loose=fell=0; max_xs=[]; times=[]
        for _ in range(n):
            mujoco.mj_resetData(m,d)
            d.qpos[0]=x_start; d.qpos[2]=GROUND_Z
            d.qpos[3:7]=[1,0,0,0]; d.qvel[:]=0.0
            d.qvel[4]=np.random.uniform(-0.02,0.02)
            mujoco.mj_forward(m,d); obs=get_obs(d); mx=0.0; end="timeout"
            for step in range(3000):
                with torch.no_grad(): t,_,_=net.get_action(obs,x_goal,deterministic=True)
                d.ctrl[0]=-float(np.clip(t,-TORQUE_MAX,TORQUE_MAX))
                mujoco.mj_step(m,d); obs=get_obs(d); x,_,th,_=obs; mx=max(mx,x)
                if abs(x)>X_DONE_LIMIT or abs(th)>TH_DONE: end="fell"; break
                if abs(x-x_goal)<0.10 and abs(th)<0.35: end="strict"; times.append(step*m.opt.timestep); break
                elif abs(x-x_goal)<0.25 and abs(th)<0.35: end="loose"; times.append(step*m.opt.timestep)
            strict+=end=="strict"; loose+=end=="loose"; fell+=end=="fell"; max_xs.append(mx)
        return {"strict_pct":strict/n*100,"loose_pct":(strict+loose)/n*100,
                "fell_pct":fell/n*100,"avg_max_x":float(np.mean(max_xs)),
                "avg_time":float(np.mean(times)) if times else 999.0}

    print(f"\n  Verifying on {len(verify_seeds)} seeds...")
    print(f"  Seed   | Strict% | Loose% | Fell% | Time"); print("  "+"-"*44)
    per_seed={}
    for vs in verify_seeds:
        r=eval_policy(net,vs); per_seed[vs]=r
        icon="OK" if r["strict_pct"]>=80 else "~" if r["strict_pct"]>=50 else "X"
        print(f"  {vs:>6} | {r['strict_pct']:>6.0f}% | {r['loose_pct']:>5.0f}% | "
              f"{r['fell_pct']:>4.0f}% | {r['avg_time']:>5.1f}s  {icon}")
    summary={"train_seed":train_seed,"verify_seeds":verify_seeds,
             "strict_pct":float(np.mean([per_seed[v]["strict_pct"] for v in verify_seeds])),
             "strict_std":float(np.std([per_seed[v]["strict_pct"] for v in verify_seeds])),
             "fell_pct":float(np.mean([per_seed[v]["fell_pct"] for v in verify_seeds])),
             "avg_time":float(np.mean([per_seed[v]["avg_time"] for v in verify_seeds
                                        if per_seed[v]["avg_time"]<999] or [999])),"per_seed":per_seed}
    print(f"\n  avg: {summary['strict_pct']:.1f}% +/- {summary['strict_std']:.1f}% strict | {summary['fell_pct']:.1f}% fell")
    logger.save(SAVE_DIR,f"cmp_seed{train_seed}",eval_results=summary)
    return net,logger,summary


## Recording Function


In [10]:
def record_all_seeds(nav_net,train_seeds=[42],verify_seeds=[788,999,555,321,444],
                     x_start=0.0,x_goal=2.0,max_steps=3000):
    model=mujoco.MjModel.from_xml_path(XML_PATH); data=mujoco.MjData(model)
    dt=model.opt.timestep; renderer=mujoco.Renderer(model,height=480,width=640)
    cam=mujoco.MjvCamera(); cam.type=mujoco.mjtCamera.mjCAMERA_FREE
    cam.lookat=np.array([1.0,0.0,0.3]); cam.distance=4.5; cam.azimuth=90; cam.elevation=-15
    for seed,role in [(s,"TRAIN") for s in train_seeds]+[(s,"VERIFY") for s in verify_seeds]:
        torch.manual_seed(seed); np.random.seed(seed)
        mujoco.mj_resetData(model,data)
        data.qpos[0]=x_start; data.qpos[2]=GROUND_Z
        data.qpos[3:7]=[1,0,0,0]; data.qvel[:]=0.0; mujoco.mj_forward(model,data)
        obs=get_obs(data); frames=[]; strict=arrived=fell=False
        for step in range(max_steps):
            with torch.no_grad(): torque,_,_=nav_net.get_action(obs,x_goal,deterministic=True)
            data.ctrl[0]=-float(np.clip(torque,-TORQUE_MAX,TORQUE_MAX))
            mujoco.mj_step(model,data); obs=get_obs(data); x,_,theta,_=obs
            fell=abs(x)>X_DONE_LIMIT or abs(theta)>TH_DONE
            arrived=abs(x-x_goal)<0.25 and abs(theta)<0.35
            strict=abs(x-x_goal)<0.10 and abs(theta)<0.35
            renderer.update_scene(data,camera=cam)
            img=PILImage.fromarray(renderer.render()); draw=ImageDraw.Draw(img); W,H=img.size
            prog=float(np.clip(x/x_goal,0,1)); bw=W-40
            pcol=(0,200,0) if strict else (255,140,0) if arrived else (30,100,220)
            draw.rectangle([20,8,W-20,28],fill=(40,40,40))
            draw.rectangle([20,8,20+int(bw*prog),28],fill=pcol)
            draw.text((22,10),"A",fill=(255,255,255)); draw.text((W-28,10),"B",fill=(255,255,255))
            draw.text((W//2-40,10),f"{prog*100:.0f}%  x={x:.3f}m",fill=(255,255,255))
            badge_col=(0,60,140) if role=="TRAIN" else (100,0,140)
            draw.rectangle([8,34,170,58],fill=badge_col)
            draw.text((12,38),f"[{role}] seed={seed}",fill=(255,255,255))
            if strict:    sc,st=(0,120,0),  f"STRICT x={x:.3f}m t={step*dt:.1f}s"
            elif arrived: sc,st=(120,100,0),f"LOOSE  x={x:.3f}m t={step*dt:.1f}s"
            elif fell:    sc,st=(140,0,0),  f"FELL   x={x:.3f}m th={np.degrees(theta):.1f}deg"
            else:         sc,st=(20,20,70), f"x={x:+.3f}m th={np.degrees(theta):+.1f}deg u={torque:+.2f}Nm t={step*dt:.1f}s"
            draw.rectangle([175,34,W-8,58],fill=sc); draw.text((178,38),st,fill=(255,255,255))
            frames.append(np.array(img))
            if fell or strict: break
        fname=f"{SAVE_DIR}/ddpg_{role.lower()}_seed{seed}.gif"
        imageio.mimsave(fname,frames,fps=30)
        status="STRICT" if strict else "LOOSE" if arrived else "FELL" if fell else "TIMEOUT"
        print(f"  [{role}] seed={seed} -> {status}  {fname}")
    print("Done.")


## Plot Episode Traces


In [11]:
def plot_episode_traces(net, algo="DDPG", x_goal=2.0, save_prefix=None):
    save_prefix=save_prefix or algo.lower()
    model=mujoco.MjModel.from_xml_path(XML_PATH); data=mujoco.MjData(model)
    dt=model.opt.timestep; mujoco.mj_resetData(model,data)
    data.qpos[0]=0.0; data.qpos[2]=GROUND_Z; data.qpos[3:7]=[1,0,0,0]; data.qvel[:]=0.0
    mujoco.mj_forward(model,data); obs=get_obs(data)
    ts,xs,ths,torqs=[],[],[],[]
    for step in range(3000):
        with torch.no_grad(): torque,_,_=net.get_action(obs,x_goal,deterministic=True)
        u=float(np.clip(torque,-TORQUE_MAX,TORQUE_MAX)); data.ctrl[0]=-u
        mujoco.mj_step(model,data); obs=get_obs(data); x,_,th,_=obs
        ts.append(step*dt); xs.append(x); ths.append(np.degrees(th)); torqs.append(torque)
        if abs(x)>X_DONE_LIMIT or abs(th)>TH_DONE or (abs(x-x_goal)<0.25 and abs(th)<0.35): break
    ts,xs,ths,torqs=np.array(ts),np.array(xs),np.array(ths),np.array(torqs)
    arrived=abs(xs[-1]-x_goal)<0.25 and abs(ths[-1])<20
    rms=np.sqrt(np.mean(torqs**2))
    fig=plt.figure(figsize=(15,9)); gs=gridspec.GridSpec(2,2,hspace=0.38,wspace=0.25)
    fig.suptitle(f"{algo} — Deterministic Episode",fontsize=13,fontweight="bold")
    ax=fig.add_subplot(gs[0,:]); ax.plot(ts,xs,"#E74C3C",lw=2,label="x position")
    ax.axhline(0,color="blue",ls=":",lw=2); ax.axhline(x_goal,color="green",ls=":",lw=2,label=f"goal {x_goal}m")
    ax.axhspan(x_goal-0.25,x_goal+0.25,alpha=0.1,color="green",label="Goal zone")
    if arrived:
        idx=np.where(np.abs(xs-x_goal)<0.25)[0][0]
        ax.axvline(ts[idx],color="green",ls="--",alpha=0.6)
        ax.annotate(f"ARRIVED t={ts[idx]:.1f}s",xy=(ts[idx],xs[idx]),
                    xytext=(ts[idx]+0.3,x_goal-0.4),color="green",fontsize=9,
                    arrowprops=dict(arrowstyle="->",color="green"))
    ax.set_xlabel("Time(s)"); ax.set_ylabel("x(m)"); ax.legend(fontsize=9); ax.grid(alpha=0.3)
    ax=fig.add_subplot(gs[1,0]); ax.plot(ts,ths,"#E74C3C",lw=2,label="theta (deg)")
    ax.axhspan(-20,20,alpha=0.06,color="green",label="Stable +-20 deg")
    ax.axhline(0,color="gray",alpha=0.4); ax.set_xlabel("Time(s)"); ax.set_ylabel("Tilt(deg)")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    ax=fig.add_subplot(gs[1,1]); ax.plot(ts,torqs,"#2ECC71",lw=2,label=f"Torque RMS={rms:.2f}Nm")
    ax.fill_between(ts,torqs,0,where=(torqs>0),alpha=0.15,color="red",label="Forward")
    ax.fill_between(ts,torqs,0,where=(torqs<0),alpha=0.15,color="blue",label="Backward")
    ax.axhline(TORQUE_MAX,color="orange",ls=":",lw=1.5); ax.axhline(-TORQUE_MAX,color="orange",ls=":",lw=1.5)
    ax.axhline(0,color="gray",alpha=0.4); ax.set_xlabel("Time(s)"); ax.set_ylabel("Torque(Nm)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.savefig(f"{save_prefix}_traces.png",dpi=150,bbox_inches="tight"); plt.show()
    print(f"  Duration {ts[-1]:.1f}s | MaxX {max(xs):.3f}m | AvgTilt {np.mean(np.abs(ths)):.1f}deg | RMS {rms:.3f}Nm")


## Plot Loss and Reward


In [12]:
def plot_loss_reward(logger=None,rewards=None,losses=None,algo="DDPG",save_prefix=None):
    save_prefix=save_prefix or algo.lower()
    if logger is not None:
        sx=[r["steps"] for r in logger.rows]; ry=[r["avg_reward"] for r in logger.rows]
        ly=[r["loss"] for r in logger.rows]; wt=getattr(logger,"total_wall_time",None); hs=True
    else: hs=False
    fig,(axr,axl)=plt.subplots(1,2,figsize=(15,5))
    title=f"{algo} Training"
    if hs and wt: title+=f"  (wall: {wt/60:.1f} min)"
    fig.suptitle(title,fontsize=13,fontweight="bold")
    if hs:
        axr.plot(sx,ry,color="#E74C3C",lw=2,label=f"avg100 (peak={max(ry):.1f})")
        axl.plot(sx,ly,color="#F39C12",lw=2,label="Critic Loss")
        axr.set_xlabel("Steps"); axl.set_xlabel("Steps")
    axr.axhline(0,color="green",ls="--",alpha=0.5); axr.set_title("Reward",fontweight="bold"); axr.legend(); axr.grid(alpha=0.3)
    axl.axhline(0,color="green",ls="--",alpha=0.5); axl.set_title("Critic Loss",fontweight="bold"); axl.legend(); axl.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_training.png",dpi=150,bbox_inches="tight"); plt.show()
    if hs and wt: print(f"Wall time: {wt/60:.1f} min")


## RUN ALL


In [13]:
net, logger, summary = train_one_verify_many("DDPG", train_seed=42, step_budget=2_000_000)



  DDPG  TRAIN seed=42

DDPG | A=0.0m -> B=2.0m | budget=2,000,000 steps
   Steps |    Ep |    Avg10 |   Avg100 |   Rec% |   Noise |    CritL
--------------------------------------------------------------------
   1,199 |    10 |   -57.77 |   -57.77 |   0.0% |   0.300 |    0.000  ..
   2,747 |    20 |   -66.14 |   -61.96 |   0.0% |   0.299 |    0.000  ..
   4,335 |    30 |   -66.34 |   -63.42 |   0.0% |   0.299 |    0.000  ..
   5,386 |    40 |   -55.40 |   -61.41 |   0.0% |   0.299 |    2.188  ..
   5,737 |    50 |   -33.41 |   -55.81 |   0.0% |   0.299 |    2.671  ..
   6,088 |    60 |   -33.19 |   -52.04 |   0.0% |   0.299 |    2.530  ..
  13,930 |    70 |  -240.51 |   -78.97 |   0.0% |   0.297 |    0.124  ..
  25,364 |    80 |  -212.98 |   -95.72 |   0.0% |   0.295 |    0.318  ..
  37,496 |    90 |  -120.65 |   -98.49 |   5.6% |   0.293 |    0.878  ..
  54,493 |   100 |   -15.38 |   -90.18 |  13.0% |   0.290 |    1.403  ..
  75,274 |   110 |  -141.28 |   -98.53 |  18.0% |   0.285 |

In [ ]:
record_all_seeds(net, train_seeds=[42], verify_seeds=[788,999,555], x_goal=2.0)


  [TRAIN] seed=42 -> STRICT  SavedSeeds/ddpg_train_seed42.gif
  [VERIFY] seed=788 -> STRICT  SavedSeeds/ddpg_verify_seed788.gif
  [VERIFY] seed=999 -> STRICT  SavedSeeds/ddpg_verify_seed999.gif
  [VERIFY] seed=555 -> STRICT  SavedSeeds/ddpg_verify_seed555.gif
Done.


: 

In [ ]:
plot_loss_reward(logger=logger, algo="DDPG")


In [ ]:
plot_episode_traces(net, algo="DDPG")
